In [0]:
-- 1. Find the preferred restaurant names for each customer
SELECT
    o.customer_id,
    r.name AS restaurant_name,
    COUNT(*) AS order_count
FROM sathvika_dinedash_db.sathvika_orders_bronze o
JOIN sathvika_dinedash_db.sathvika_restaurants_bronze r
    ON o.restaurant_id = r.restaurant_id
GROUP BY
    o.customer_id,
    r.name
ORDER BY
    o.customer_id,
    order_count DESC;

customer_id,restaurant_name,order_count
C0001,Spice Villa Grill,2
C0001,Curry Leaf Hub,1
C0002,null,1
C0003,Spice Villa Grill,1
C0004,Pasta Palace Kitchen,1
C0004,Tasty Bites Corner,1
C0004,null,1
C0006,Pasta Palace Grill,1
C0007,Tasty Bites Hub,2
C0007,Spice Villa Grill,1


In [0]:
-- 2. Find the top 10 areas receiving orders in each month
SELECT month, area_name, order_count
FROM (
    SELECT
        MONTH(o.timestamp) AS month,
        l.area AS area_name,
        COUNT(*) AS order_count,
        ROW_NUMBER() OVER (
            PARTITION BY MONTH(o.timestamp)
            ORDER BY COUNT(*) DESC
        ) AS rn
    FROM sathvika_dinedash_db.sathvika_orders_bronze o
    JOIN sathvika_dinedash_db.sathvika_locations_bronze l
        ON o.delivery_location_id = l.location_id
    GROUP BY MONTH(o.timestamp), l.area
)
WHERE rn <= 10
ORDER BY month, order_count DESC;

month,area_name,order_count
1,Rodriguez Mill,87
1,Eric Hills,86
1,Hill Station,80
1,Smith Estates,79
1,Harper Street,76
1,Morgan Key,75
1,Copeland Parks,75
1,John Underpass,74
1,Ashley Extensions,73
1,Jennifer Wall,73


In [0]:
-- 3. Find all successfully delivered orders that were paid using a card, to analyze payment preferences and delivery success rates
SELECT
    order_id,
    customer_id,
    restaurant_id,
    payment_method,
    order_status,
    total_amount
FROM sathvika_dinedash_db.sathvika_orders_bronze
WHERE order_status = 'delivered'
  AND payment_method = 'card';

order_id,customer_id,restaurant_id,payment_method,order_status,total_amount
e293e41d-e335-4d6b-8504-34f96140e660,C5605,R006,card,delivered,21.2
c8bd7ee7-a451-447a-811b-fd69aff36062,C7525,R053,card,delivered,75.97
e71e1b46-c026-4913-9b58-870fde58e72b,C1909,R047,card,delivered,113.32
ede3312f-d227-42a5-a1d0-d191999abbd1,C3246,R136,card,delivered,112.3
7e5fb9b0-c7e0-4080-89f3-6707df2a16a4,C8344,R074,card,delivered,70.4
22accf1f-f43d-4051-9240-9a971c27bf85,C3024,R039,card,delivered,54.82
bee913bc-f4fd-44b7-8f51-229c93ef4c5d,C2294,R018,card,delivered,18.34
5d838bcb-e367-4734-93fe-c9a2551e440f,C8811,R190,card,delivered,68.42
938839a3-20ac-4b3b-a9b1-fb9b364bc210,C1179,R159,card,delivered,51.5
3cd888ae-1614-4d75-bd5c-037edd3684f4,C2504,R110,card,delivered,118.6


In [0]:
-- 4. Which cuisines generate the highest average order value?
-- Hint: Join orders, restaurants, group by cuisine, and calculate average total_amount.
SELECT
    r.cuisines,
    ROUND(AVG(o.total_amount), 2) AS average_order_value
FROM sathvika_dinedash_db.sathvika_orders_bronze o
JOIN sathvika_dinedash_db.sathvika_restaurants_bronze r
    ON o.restaurant_id = r.restaurant_id
GROUP BY r.cuisines
ORDER BY average_order_value DESC;

cuisines,average_order_value
"Thai,Chinese",160.1
"Thai,Italian,Chinese",156.68
"Mexican,American",146.19
"Thai,Mexican",144.46
"Mexican,American,Thai",143.95
"Thai,Indian,American",129.59
"American,Mexican",123.96
"Thai,American,Mexican",121.78
"Italian,Indian,Mexican",121.05
"American,Chinese",118.95


In [0]:
-- 5. Which delivery agents have delivered the most high-value orders (above $50)?
-- Hint: Filter orders for total_amount > 50, join with agent, group by agent_id, count orders.

SELECT
    o.agent_id,
    a.name,
    COUNT(*) AS order_count
FROM sathvika_dinedash_db.sathvika_orders_bronze o
JOIN sathvika_dinedash_db.sathvika_delivery_agents_bronze a
    ON o.agent_id = a.agent_id
WHERE o.total_amount > 50
GROUP BY o.agent_id, a.name
ORDER BY order_count DESC;

agent_id,name,order_count
A309,Christopher Chase,213
A207,Christopher Williams,171
A393,Justin Lam,118
A350,Charlene Morgan,112
A370,Joan Harrison,110
A138,Carrie Maxwell,108
A178,John King,108
A360,Justin Smith,106
A168,James Osborn,104
A245,Teresa Ryan,102


In [0]:
-- 6. Which cities (area_name) are generating the most revenue for Italian restaurants?
-- Hint: Join orders, restaurants, locations, filter cuisine = 'Italian', group by area_name, sum total_amount.

SELECT
    l.area AS area_name,
    ROUND(SUM(o.total_amount), 2) AS revenue
FROM sathvika_dinedash_db.sathvika_orders_bronze o
JOIN sathvika_dinedash_db.sathvika_restaurants_bronze r
    ON o.restaurant_id = r.restaurant_id
JOIN sathvika_dinedash_db.sathvika_locations_bronze l
    ON r.location_id = l.location_id
WHERE r.cuisines LIKE '%Italian%'
GROUP BY l.area
ORDER BY revenue DESC;

area_name,revenue
Padilla Mall,27277.6
Daryl Extensions,17360.82
Brennan Center,16184.11
Harris Grove,16109.65
Boyle Dam,15221.04
Montgomery Gateway,14807.66
Smith Rapids,11402.97
Torres Mountain,11368.6
Henderson Creek,11356.79
Rich Expressway,10652.45


In [0]:
-- 7. What is the tip-to-total ratio by cuisine type?
-- Hint: Join orders, restaurants, group by cuisine, calculate average tip / total_amount.

SELECT
    r.cuisines,
    SUM(o.tip) / SUM(o.total_amount) AS tip_to_total_ratio
FROM sathvika_dinedash_db.sathvika_orders_bronze o
JOIN sathvika_dinedash_db.sathvika_restaurants_bronze r
    ON o.restaurant_id = r.restaurant_id
GROUP BY r.cuisines
ORDER BY tip_to_total_ratio DESC;

cuisines,tip_to_total_ratio
"Chinese,American",0.1630295032439566
"Indian,Mexican,American",0.1433364926011985
"Thai,American,Indian",0.09289116111708078
"Chinese,Italian,Thai",0.08002964201565706
"Mexican,Italian,American",0.07546926003361999
"American,Italian,Indian",0.06287668787668786
"Italian,Thai,Chinese",0.05765953938327928
"Italian,Chinese,American",0.05066045436142366
"Chinese,Indian,American",0.04965877495997978
"Italian,Mexican",0.0452337176353529


In [0]:
-- 8. Which menu items are most frequently ordered, and from which restaurants?
-- Hint: Explode items_ordered in orders, join with menu_items, group by item_name, restaurant_id, count.

SELECT
    m.item_name,
    sub.restaurant_id,
    COUNT(*) AS order_count
FROM (
    SELECT o.restaurant_id, item.item_id
    FROM sathvika_dinedash_db.sathvika_orders_bronze o
    LATERAL VIEW EXPLODE(o.items_ordered) t AS item
) sub
JOIN sathvika_dinedash_db.sathvika_menu_items_bronze m
    ON sub.item_id = m.item_id
GROUP BY m.item_name, sub.restaurant_id
ORDER BY order_count DESC;

item_name,restaurant_id,order_count
Argue Cake,R198,111
Purpose Brownie,R138,104
Even Tea,R090,103
Rich Biryani,R139,101
Painting Coffee,R084,101
Sell Ice Cream,R039,101
Others Wrap,R165,100
Finish Garlic Bread,R082,100
Market Spring Roll,R102,99
Student Tea,R158,99


In [0]:
-- 9. Which agents have delivered the most diverse cuisines?
-- Hint: Join orders, restaurants, group by agent_id, count distinct cuisine.

SELECT
    o.agent_id,
    COUNT(DISTINCT r.cuisines) AS cuisine_count
FROM sathvika_dinedash_db.sathvika_orders_bronze o
JOIN sathvika_dinedash_db.sathvika_restaurants_bronze r
    ON o.restaurant_id = r.restaurant_id
GROUP BY o.agent_id
ORDER BY cuisine_count DESC;

agent_id,cuisine_count
A309,49
A168,45
A360,44
A245,44
A207,44
A393,43
A351,42
A331,42
A355,41
A258,41


In [0]:
-- 10. What is the average customer lifetime value by signup year?
-- Hint: Join orders, customers, extract year from signup_date, group by year, sum total_amount per customer, then average total amount per customer, then average.

SELECT
    signup_year,
    ROUND(AVG(customer_total), 2) AS average_customer_lifetime_value
FROM (
    SELECT
        c.customer_id,
        YEAR(c.signup_date) AS signup_year,
        SUM(o.total_amount) AS customer_total
    FROM sathvika_dinedash_db.sathvika_orders_bronze o
    JOIN sathvika_dinedash_db.sathvika_customers_bronze c
        ON o.customer_id = c.customer_id
    GROUP BY c.customer_id, YEAR(c.signup_date)
)
GROUP BY signup_year
ORDER BY signup_year;

signup_year,average_customer_lifetime_value
1954,22.64
1955,172.65
1956,139.9
1957,155.41
1958,135.53
1959,141.08
1960,173.02
1961,222.25
1962,192.06
1963,137.26


In [0]:
-- 11. Which restaurants have the highest order frequency per location?
-- Hint: Join orders, restaurants, locations, group by restaurant_id, location_id, count orders.

SELECT
    o.restaurant_id,
    r.location_id,
    l.area,
    COUNT(*) AS order_count
FROM sathvika_dinedash_db.sathvika_orders_bronze o
JOIN sathvika_dinedash_db.sathvika_restaurants_bronze r
    ON o.restaurant_id = r.restaurant_id
JOIN sathvika_dinedash_db.sathvika_locations_bronze l
    ON r.location_id = l.location_id
GROUP BY
    o.restaurant_id,
    r.location_id,
    l.area
ORDER BY order_count DESC;

restaurant_id,location_id,area,order_count
R005,L039,Elizabeth Locks,114
R198,L022,White Tunnel,111
R031,L015,Hunt Drives,110
R083,L069,Reed Glens,108
R171,L093,Copeland Parks,108
R131,L046,Reynolds Curve,108
R058,L031,Edwards Plains,107
R092,L057,Kimberly Points,106
R138,L018,Hill Station,104
R151,L065,Nicholas Fork,104


In [0]:
-- 12. Which locations are prone to order cancellations?
-- Hint: Filter orders with order_status = 'cancelled', join locations using delivery_location_id, group by area_name, count.

SELECT
    l.area AS area_name,
    COUNT(*) AS cancelled_orders
FROM sathvika_dinedash_db.sathvika_orders_bronze o
JOIN sathvika_dinedash_db.sathvika_locations_bronze l
    ON o.delivery_location_id = l.location_id
WHERE o.order_status = 'cancelled'
GROUP BY l.area
ORDER BY cancelled_orders DESC;

area_name,cancelled_orders
Evans Stream,58
Elizabeth Locks,57
Flores Vista,55
Ashley Extensions,53
Padilla Mall,52
Hill Station,52
Curtis Oval,51
White Tunnel,51
Rodriguez Mill,51
Joshua Drive,51


In [0]:
-- 13. Find top 10 customers who have ordered from the widest variety of restaurants.
-- Hint: Join orders, group by customer_id, count distinct restaurant_id, sort and pick top 10.

SELECT
    customer_id,
    COUNT(DISTINCT restaurant_id) AS restaurant_count
FROM sathvika_dinedash_db.sathvika_orders_bronze
GROUP BY customer_id
ORDER BY restaurant_count DESC
LIMIT 10;

customer_id,restaurant_count
C7381,9
C6444,8
C2231,8
C0362,8
C1449,8
C6672,7
C6971,7
C4867,7
C1423,7
C9530,7


In [0]:
-- 14. Identify restaurants with frequent repeat customers.
-- Hint: Join orders, group by restaurant_id, customer_id, count, filter where count > 1.

SELECT
    restaurant_id,
    customer_id,
    COUNT(*) AS order_count
FROM sathvika_dinedash_db.sathvika_orders_bronze
GROUP BY restaurant_id, customer_id
HAVING COUNT(*) > 1
ORDER BY order_count DESC;

restaurant_id,customer_id,order_count
R006,C5296,3
R148,C7659,2
R138,C0957,2
R151,C3027,2
R188,C0579,2
R055,C9140,2
R005,C7346,2
R038,C8626,2
R075,C7228,2
R126,C4980,2


In [0]:
-- 15. Which payment methods are most preferred for high-value orders across cuisines?
-- Hint: Filter orders with total_amount > 40, join restaurants, group by cuisine, payment_method, count.

SELECT
    r.cuisines,
    o.payment_method,
    COUNT(*) AS order_count
FROM sathvika_dinedash_db.sathvika_orders_bronze o
JOIN sathvika_dinedash_db.sathvika_restaurants_bronze r
    ON o.restaurant_id = r.restaurant_id
WHERE o.total_amount > 40
GROUP BY r.cuisines, o.payment_method
ORDER BY order_count DESC;

cuisines,payment_method,order_count
Indian,card,267
Indian,wallet,245
Indian,cash,241
Indian,UPI,239
Thai,card,209
Mexican,wallet,208
Mexican,cash,199
Mexican,card,189
Thai,wallet,188
Mexican,UPI,186
